In [1]:
from semantic_kernel import Kernel
from dotenv import load_dotenv
import os
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion


In [2]:
load_dotenv()

API_KEY     = os.environ["OPENAI_API_KEY"]
ENDPOINT    = os.environ["OPENAI_ENDPOINT"]
DEPLOYMENT  = os.environ.get("OPENAI_DEPLOYMENT")
API_VERSION = os.environ.get("OPENAI_API_VERSION")

In [3]:
client = AzureChatCompletion( 
    deployment_name=DEPLOYMENT,
    endpoint=ENDPOINT,  
    api_key=API_KEY,
    api_version=API_VERSION
)

In [85]:
from typing import Annotated
from semantic_kernel.functions import kernel_function

class LightsPlugin:
    lights = [
        {"id": 1, "name": "Table Lamp", "is_on": False},
        {"id": 2, "name": "Porch Light", "is_on": False},
        {"id": 3, "name": "Chandelier", "is_on": True},
    ]
    
    @kernel_function(description="Get a list of lights and their current state", 
                     name = "get_lights")
    def get_lights(self)-> str:
        """Returns a list of lights and their current state"""
        return self.lights
    
    @kernel_function(name= "change_state", description="Changes the state of a light with given id and desired state")
    def change_state(self, 
                    id: Annotated[int, "The id of the light to change"],
                    is_on: Annotated[bool, "True to turn on, False to turn off"],
                     )-> str:
        """
        Changes the state of a light given its id and the desired state
        
        Parameters:
            id (int): The id of the light to change
            is_on (bool): The desired state of the light (True for on, False for off)
        """
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return f"{light['name']} is now {'on' if is_on else 'off'}"
        return "Light not found"
    

In [91]:
from semantic_kernel.contents import ChatHistory
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior


# --- Kernel setup ---
kernel = Kernel()
kernel.add_service(client)  # or your existing client
kernel.add_plugin(plugin=LightsPlugin(), plugin_name="lightsTool")

# --- Chat history ---
chat_history = ChatHistory()
chat_history.add_system_message(
    "You are a helpful assistant for managing lights in a house. "
    "Use the available tools to list lights and change their state."
    "always return a then end the actual state of every light after any change state action."
)

# --- Execution settings with auto function calling ---
settings = OpenAIChatPromptExecutionSettings(
    service_id="default",
    function_choice_behavior=FunctionChoiceBehavior.Auto(),
)

# --- Invoke ---
user_query = "Change the state of the Table Lamp to on."
chat_history.add_user_message(user_query)
print(f"User query: {user_query}")

chat_service = kernel.get_service(service_id="default")

response = await chat_service.get_chat_message_content(
    chat_history=chat_history,
    settings=settings,
    kernel=kernel,
)

print(f"Agent: {response}")

User query: Change the state of the Table Lamp to on.
Agent: The Table Lamp is already on. 

Current state of the lights:
1. Table Lamp: On
2. Porch Light: Off
3. Chandelier: On


In [92]:
user_query = "list all the lights and their state"
chat_history.add_user_message(user_query)
print(f"User query: {user_query}")

chat_service = kernel.get_service(service_id="default")

response = await chat_service.get_chat_message_content(
    chat_history=chat_history,
    settings=settings,
    kernel=kernel,
)

print(f"Agent: {response}")

User query: list all the lights and their state
Agent: Here is the list of lights and their current state:

1. **Table Lamp**: On
2. **Porch Light**: Off
3. **Chandelier**: On


In [ ]:
# # run task with kernel
# kernel = Kernel()

# kernel.add_service(client)

# # register lights plugin
# kernel.add_plugin(plugin=LightsPlugin(), plugin_name="lightsTool")

# # define and register the semantic function
# lightsmanager_prompt= """You are a helpful assistant for managing the state of the lights in a house. 

# You have access those lights: {{lightsTool.get_lights}}. Use it to list every light and its state.

# To change a light state:
# - Extract the light id integer as `id`
# - Extract the desired state as `is_on`
# - Then call: {{lightsTool.change_state id=$id is_on=$is_on}}

# After executing the tool call, explain the result to the user.
# """

# # this is Function Chaining: the LLM is instructed to excute the tool call first, and then use the output to answer the user query
# light_manager_function = kernel.add_function(function_name="light_manager_function", plugin_name="lightsTool", prompt=lightsmanager_prompt)






In [90]:
# # user input: "return lights state
# user_query = "What is the state of every lights?"
# print(f"User query: {user_query} ")

# result = await kernel.invoke(function=light_manager_function, input =user_query) 
# print(f"Agent: {result.value[0].items[0].text}")